# Open Tennis Data v3 with Polars

This notebook queries the v3 Parquet release lazily with Polars. It covers results, fixtures, players, tournaments, integrity checks, provenance, coverage, health, source policies, and quarantine records.

Use an immutable release tag while v3 remains preview. The `latest` channel should be used only when its manifest reports `release_status=stable`.

## Setup

From a repository checkout, install the optional notebook dependencies:

```bash
python -m pip install '.[notebook]'
```

Set `OPEN_TENNIS_DATA_RELEASE` to an immutable tag, or set `OPEN_TENNIS_DATA_RELEASE_DIR` to a local directory containing the 13 verified release payloads. The local directory option is useful for testing a staged release before publication.

In [ ]:
from __future__ import annotations

import json
import os
import urllib.request
from datetime import date, timedelta
from pathlib import Path

import polars as pl

REPOSITORY = os.getenv("OPEN_TENNIS_DATA_REPOSITORY", "ryantjx/tennis-match-data")
RELEASE = os.getenv("OPEN_TENNIS_DATA_RELEASE", "latest")
LOCAL_RELEASE_DIR = os.getenv("OPEN_TENNIS_DATA_RELEASE_DIR")

if LOCAL_RELEASE_DIR:
    release_dir = Path(LOCAL_RELEASE_DIR).expanduser().resolve()
    manifest = json.loads((release_dir / "manifest.json").read_text(encoding="utf-8"))
    asset_urls = {item["name"]: str(release_dir / item["name"]) for item in manifest["assets"]}
else:
    route = "latest/download" if RELEASE == "latest" else f"download/{RELEASE}"
    manifest_url = f"https://github.com/{REPOSITORY}/releases/{route}/manifest.json"
    request = urllib.request.Request(manifest_url, headers={"User-Agent": "open-tennis-data-polars"})
    with urllib.request.urlopen(request, timeout=60) as response:
        manifest = json.load(response)
    asset_urls = {item["name"]: item["url"] for item in manifest["assets"]}

def asset(name: str) -> str:
    """Return a local path or immutable release URL for one asset."""
    return asset_urls[name]

{
    "release_tag": manifest["release_tag"],
    "release_status": manifest["release_status"],
    "as_of": manifest["as_of"],
    "preview_reasons": manifest["preview_reasons"],
}

## Lazy release scans

`scan_parquet` keeps the queries lazy, allowing Polars to push projections and filters into Parquet scans. For remote URLs, only the required columns and row groups need to be read.

In [ ]:
completed = pl.scan_parquet(asset("completed.parquet"))
fixtures = pl.scan_parquet(asset("fixtures.parquet"))
all_matches = pl.scan_parquet(asset("matches.parquet"))
tournaments = pl.scan_parquet(asset("tournaments.parquet"))
players = pl.scan_parquet(asset("players.parquet"))
provenance = pl.scan_parquet(asset("provenance.parquet"))
sources = pl.scan_parquet(asset("sources.parquet"))
coverage = pl.scan_parquet(asset("coverage.parquet"))
health = pl.scan_parquet(asset("health.parquet"))
quarantine = pl.scan_parquet(asset("quarantine.parquet"))
catalog = pl.scan_parquet(asset("catalog.parquet"))

catalog.sort("path").collect(engine="streaming")

## Counts by tour, season, and status

In [ ]:
(
    all_matches
    .group_by("tour", "year", "status")
    .agg(pl.len().alias("matches"))
    .sort("year", "tour", "status")
    .collect(engine="streaming")
)

## Recent results

In [ ]:
(
    completed
    .select(
        "date",
        "tour",
        "tournament_name",
        "round",
        "player1_name",
        "player2_name",
        "status",
        "score",
    )
    .sort("date", descending=True, nulls_last=True)
    .head(50)
    .collect(engine="streaming")
)

## Upcoming and incomplete fixtures

In [ ]:
today = date.today()
next_two_weeks = today + timedelta(days=14)

upcoming = (
    fixtures
    .filter(pl.col("date").is_between(today, next_two_weeks, closed="both"))
    .select(
        "date",
        "tour",
        "tournament_name",
        "round",
        "player1_name",
        "player2_name",
    )
    .sort("date", "tour", "tournament_name", "round")
)

awaiting_schedule = (
    fixtures
    .filter(
        pl.col("date").is_null()
        | pl.col("player1_name").is_null()
        | pl.col("player2_name").is_null()
    )
    .select(
        "match_id",
        "tour",
        "tournament_name",
        "round",
        "date",
        "player1_name",
        "player2_name",
    )
    .sort("tour", "tournament_name", "round", "match_id")
)

upcoming.collect(engine="streaming"), awaiting_schedule.collect(engine="streaming")

## Player search

In [ ]:
def player_name_contains(column: str, value: str) -> pl.Expr:
    return (
        pl.col(column)
        .list.join(" / ", ignore_nulls=True)
        .str.to_lowercase()
        .str.contains(value.casefold(), literal=True)
        .fill_null(False)
    )

player_query = "Alcaraz"
(
    all_matches
    .filter(
        player_name_contains("player1_name", player_query)
        | player_name_contains("player2_name", player_query)
    )
    .select(
        "date",
        "tour",
        "tournament_name",
        "round",
        "player1_name",
        "player2_name",
        "status",
        "score",
    )
    .sort("date", descending=True, nulls_last=True)
    .collect(engine="streaming")
)

## Head-to-head results

In [ ]:
def head_to_head(player_a: str, player_b: str) -> pl.DataFrame:
    a_on_side_1 = player_name_contains("player1_name", player_a)
    a_on_side_2 = player_name_contains("player2_name", player_a)
    b_on_side_1 = player_name_contains("player1_name", player_b)
    b_on_side_2 = player_name_contains("player2_name", player_b)
    return (
        completed
        .filter((a_on_side_1 & b_on_side_2) | (a_on_side_2 & b_on_side_1))
        .select(
            "date",
            "tournament_name",
            "round",
            "player1_name",
            "player2_name",
            "winner_id",
            "status",
            "score",
        )
        .sort("date")
        .collect(engine="streaming")
    )

head_to_head("Sinner", "Alcaraz")

## Tournament summaries

In [ ]:
(
    completed
    .join(tournaments, on=["tournament_id", "tour", "year"], how="inner")
    .group_by("tour", "year", "level", "tournament_name")
    .agg(
        pl.col("date").min().alias("first_match_date"),
        pl.col("date").max().alias("last_match_date"),
        pl.len().alias("matches"),
    )
    .sort("year", "tour", "first_match_date", descending=[True, False, False])
    .collect(engine="streaming")
)

## Player identity lookup

In [ ]:
(
    players
    .filter(pl.col("name").str.to_lowercase().str.contains("swiatek", literal=True))
    .sort("tour", "name", "player_id")
    .collect(engine="streaming")
)

## Public-data integrity checks

Every value in the result should be zero.

In [ ]:
match_identity = all_matches.select(
    pl.len().alias("rows"),
    pl.col("match_id").n_unique().alias("unique_match_ids"),
).collect(engine="streaming")

terminal_without_date = (
    completed
    .filter(pl.col("date").is_null() | (pl.col("status") == "fixture"))
    .select(pl.len())
    .collect(engine="streaming")
    .item()
)
fixture_with_result = (
    fixtures
    .filter(
        (pl.col("status") != "fixture")
        | pl.col("winner_id").is_not_null()
        | pl.col("score").is_not_null()
    )
    .select(pl.len())
    .collect(engine="streaming")
    .item()
)
out_of_scope = (
    all_matches
    .filter(
        (pl.col("year") < 2020)
        | (pl.col("draw") != "main")
        | (pl.col("format") != "singles")
    )
    .select(pl.len())
    .collect(engine="streaming")
    .item()
)

integrity = pl.DataFrame(
    {
        "check": [
            "duplicate_match_ids",
            "terminal_without_date",
            "fixture_with_result",
            "out_of_scope",
        ],
        "invalid_rows": [
            match_identity["rows"][0] - match_identity["unique_match_ids"][0],
            terminal_without_date,
            fixture_with_result,
            out_of_scope,
        ],
    }
)
integrity

## Exact-date provenance check

In [ ]:
accepted_dates = (
    provenance
    .filter(
        (pl.col("observation_kind") == "match_date")
        & (pl.col("date_role") == "played")
        & (pl.col("date_precision") == "day")
    )
    .select(
        "match_id",
        "tour",
        "year",
        pl.col("played_on").alias("date"),
    )
    .unique()
)

(
    completed
    .join(accepted_dates, on=["match_id", "tour", "year", "date"], how="anti")
    .select(pl.len().alias("terminal_rows_missing_date_evidence"))
    .collect(engine="streaming")
)

## Coverage and stable-release blockers

In [ ]:
coverage_blockers = (
    coverage
    .filter(
        (pl.col("coverage_status") != "complete")
        | (pl.col("missing_tournament_rows").fill_null(0) > 0)
        | (pl.col("missing_match_rows").fill_null(0) > 0)
        | (pl.col("missing_date_rows").fill_null(0) > 0)
        | (pl.col("source_conflicts").fill_null(0) > 0)
    )
    .sort("year", "tour", "level", "lifecycle")
)

missing_retrieval_times = (
    provenance
    .filter(pl.col("retrieved_at").is_null())
    .group_by("tour", "year", "observation_kind")
    .agg(pl.len().alias("observations"))
    .sort("year", "tour", "observation_kind")
)

coverage_blockers.collect(engine="streaming"), missing_retrieval_times.collect(engine="streaming")

## Release health

In [ ]:
(
    health
    .with_columns((pl.lit(date.today()) - pl.col("as_of")).dt.total_hours().alias("age_hours"))
    .sort("tour")
    .collect(engine="streaming")
)

## Source policies and attribution

In [ ]:
(
    sources
    .select(
        "policy_source",
        "policy_state",
        "terms_url",
        "allowed_uses",
        "attribution",
        "reviewed_at",
        "policy_revision",
    )
    .unique()
    .sort("policy_source")
    .collect(engine="streaming")
)

## Quarantine summary

In [ ]:
(
    quarantine
    .group_by("tour", "year", "reason")
    .agg(pl.len().alias("rows"))
    .sort("year", "tour", "rows", descending=[True, False, True])
    .collect(engine="streaming")
)

## Save a filtered extract

Uncomment the last line to write a deterministic analysis subset. Generated extracts should not be committed to the repository.

In [ ]:
wta_2025 = (
    all_matches
    .filter((pl.col("tour") == "wta") & (pl.col("year") == 2025))
    .sort("date", "tournament_id", "round", "match_id", nulls_last=True)
)

wta_2025.select(pl.len().alias("rows")).collect(engine="streaming")
# wta_2025.collect(engine="streaming").write_parquet("wta-2025.parquet", compression="zstd")